## Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

## Cell 2: Load Data

In [3]:
stock_prices = pd.read_csv("../data/stock_prices.csv", index_col=0, parse_dates=True)
etf_prices = pd.read_csv("../data/etf_prices.csv", index_col=0, parse_dates=True)
stock_returns = pd.read_csv(
    "../data/stock_returns_log.csv", index_col=0, parse_dates=True
)
etf_returns = pd.read_csv("../data/etf_returns_log.csv", index_col=0, parse_dates=True)
stock_meta = pd.read_csv("../data/stock_meta.csv", index_col=0)

print(f"Stock prices:  {stock_prices.shape}")
print(f"ETF prices:    {etf_prices.shape}")
print(f"Stock returns: {stock_returns.shape}")
print(f"ETF returns:   {etf_returns.shape}")
print(f"Stock meta:    {stock_meta.shape}")

Stock prices:  (1509, 410)
ETF prices:    (1509, 15)
Stock returns: (1508, 410)
ETF returns:   (1508, 15)
Stock meta:    (505, 3)


## Cell 3: Set Parameters

In [ ]:
WINDOW = 60  # days — rolling window for regression and OU fitting
KAPPA_MIN = 252 / 30  # = 8.4 — minimum speed of mean reversion to trade
S_OPEN = 1.25  # open a trade when |s-score| exceeds this
S_CLOSE_L = 0.50  # close a long position when s-score rises above this
S_CLOSE_S = 0.75  # close a short position when s-score falls below this

## Cell 4: ETF regression 

In [ ]:
def regress_on_etf(stock_ret, etf_ret):
    """
    Regress one stock's return series on its assigned ETF returns.
    Implements equation (11) from the paper:
        R_i = alpha_i + beta_i * R_ETF + residual

    Parameters
    ----------
    stock_ret : pd.Series, shape (60,)
        Log returns for one stock over the estimation window
    etf_ret : pd.Series, shape (60,)
        Log returns for the corresponding sector ETF

    Returns
    -------
    alpha : float
        Intercept — drift of the idiosyncratic component
    beta : float
        Slope — sensitivity of the stock to its ETF
    X : np.array, shape (60,)
        Cumulative sum of residuals — the cointegration residual process
    """
    X_reg = etf_ret.values.reshape(-1, 1)
    y = stock_ret.values

    model = LinearRegression(fit_intercept=True).fit(X_reg, y)

    alpha = model.intercept_
    beta = model.coef_[0]

    residuals = y - model.predict(X_reg)

    X = np.cumsum(residuals)

    return alpha, beta, X